# Mutual Fund Analytics — Day 3: Exploratory Data Analysis
### Bluestock Fintech Internship | June 2026

**Objective:** Comprehensive EDA on all 10 cleaned mutual fund datasets.  
**Charts:** 18 visualizations | **Tools:** Pandas, NumPy, Plotly, Seaborn, Matplotlib  
**Data:** `data/processed/` — 10 cleaned CSVs | `data/db/bluestock_mf.db`

In [1]:
# Cell 1 — Imports & Configuration
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# Paths
PROJECT_ROOT  = Path(".").resolve().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CHARTS_DIR    = PROJECT_ROOT / "reports" / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

# Style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 150, "figure.figsize": (12, 6)})
PLOTLY_TEMPLATE = "plotly_white"

print(f"Project root : {PROJECT_ROOT}")
print(f"Charts dir   : {CHARTS_DIR}")

Project root : D:\Bluestock-internship
Charts dir   : D:\Bluestock-internship\reports\charts


In [2]:
# Cell 2 — Load All Cleaned Datasets
fund_master  = pd.read_csv(PROCESSED_DIR / "01_fund_master_clean.csv")
nav_history  = pd.read_csv(PROCESSED_DIR / "02_nav_history_clean.csv", parse_dates=["date"])
aum_data     = pd.read_csv(PROCESSED_DIR / "03_aum_by_fund_house_clean.csv", parse_dates=["date"])
sip_inflows  = pd.read_csv(PROCESSED_DIR / "04_monthly_sip_inflows_clean.csv", parse_dates=["month"])
cat_inflows  = pd.read_csv(PROCESSED_DIR / "05_category_inflows_clean.csv", parse_dates=["month"])
folio_count  = pd.read_csv(PROCESSED_DIR / "06_industry_folio_count_clean.csv", parse_dates=["month"])
performance  = pd.read_csv(PROCESSED_DIR / "07_scheme_performance_clean.csv")
transactions = pd.read_csv(PROCESSED_DIR / "08_investor_transactions_clean.csv", parse_dates=["transaction_date"])
portfolio    = pd.read_csv(PROCESSED_DIR / "09_portfolio_holdings_clean.csv")
benchmark    = pd.read_csv(PROCESSED_DIR / "10_benchmark_indices_clean.csv", parse_dates=["date"])

# Normalize transaction_type casing
transactions["transaction_type"] = transactions["transaction_type"].str.strip().str.title()
transactions["transaction_type"] = transactions["transaction_type"].replace({"Sip": "SIP"})

datasets = {
    "fund_master": fund_master, "nav_history": nav_history,
    "aum_data": aum_data, "sip_inflows": sip_inflows,
    "cat_inflows": cat_inflows, "folio_count": folio_count,
    "performance": performance, "transactions": transactions,
    "portfolio": portfolio, "benchmark": benchmark,
}
print(f"{'Dataset':<20} {'Rows':>8} {'Cols':>6}")
print("-" * 40)
for name, df in datasets.items():
    print(f"{name:<20} {len(df):>8,} {df.shape[1]:>6}")

Dataset                  Rows   Cols
----------------------------------------
fund_master                40     15
nav_history            46,000      5
aum_data                   90      5
sip_inflows                48      6
cat_inflows               144      3
folio_count                21      6
performance                40     24
transactions           32,778     16
portfolio                 322      8
benchmark               8,050      3


---
## Chart 1 — NAV Trend Analysis
**Business Objective:** Track daily NAV movements for all schemes 2022–2026.

In [ ]:
# Chart 1 — Interactive NAV Trend (Daily)
nav_merged = nav_history.merge(
    fund_master[["amfi_code", "scheme_name", "fund_house", "category"]],
    on="amfi_code", how="left"
)
nav_merged["short_name"] = (
    nav_merged["scheme_name"]
    .str.replace(r" - (Regular|Direct) Plan.*", "", regex=True)
    .str.slice(0, 35)
)

fig1 = px.line(
    nav_merged, x="date", y="nav", color="short_name",
    title="📈 Daily NAV Trend — All Mutual Fund Schemes (2022–2026)",
    labels={"nav": "NAV (₹)", "date": "Date", "short_name": "Scheme"},
    template=PLOTLY_TEMPLATE,
    hover_data={"fund_house": True, "category": True},
)
fig1.add_annotation(x="2023-12-01", y=nav_merged["nav"].max() * 0.85,
    text="🚀 2023 Bull Run", showarrow=False, font=dict(color="green", size=12))
fig1.add_annotation(x="2024-06-01", y=nav_merged["nav"].max() * 0.60,
    text="📉 2024 Correction", showarrow=False, font=dict(color="red", size=12))
fig1.update_layout(height=600, legend=dict(orientation="v", x=1.01, font=dict(size=9)))
fig1.write_image(str(CHARTS_DIR / "01_nav_trend.png"), width=1400, height=600)
fig1.show()
print(f"✅ Saved: 01_nav_trend.png | Total points: {len(nav_merged):,}")

In [ ]:
# Chart 2 — AUM Growth by Fund House (Seaborn Grouped Bar)
aum_data["year"] = aum_data["date"].dt.year
aum_yearly = aum_data.groupby(["year", "fund_house"])["aum_lakh_crore"].mean().reset_index()

fig2, ax = plt.subplots(figsize=(14, 7))
sns.barplot(data=aum_yearly, x="fund_house", y="aum_lakh_crore", hue="year", palette="muted", ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt="₹%.1fL", fontsize=7, padding=2, rotation=90)
ax.set_title("AUM Growth by Fund House (2022–2025)\nSBI Leads at ₹12.5 Lakh Crore", fontsize=14, fontweight="bold")
ax.set_xlabel("Fund House")
ax.set_ylabel("Average AUM (₹ Lakh Crore)")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Year", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig(CHARTS_DIR / "02_aum_growth.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: 02_aum_growth.png")

In [ ]:
# Chart 3 — SIP Inflow Time Series (Plotly)
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=sip_inflows["month"], y=sip_inflows["sip_inflow_crore"],
    mode="lines+markers", name="Monthly SIP Inflow",
    line=dict(color="#2196F3", width=2.5),
    fill="tozeroy", fillcolor="rgba(33,150,243,0.1)",
    hovertemplate="<b>%{x|%b %Y}</b><br>SIP Inflow: ₹%{y:,.0f} Cr<extra></extra>",
))
max_row = sip_inflows.loc[sip_inflows["sip_inflow_crore"].idxmax()]
fig3.add_annotation(
    x=max_row["month"], y=max_row["sip_inflow_crore"],
    text=f"🏆 All-Time High<br>₹{max_row['sip_inflow_crore']:,} Cr",
    showarrow=True, arrowhead=2, arrowcolor="red",
    font=dict(color="red", size=12), bgcolor="white", bordercolor="red",
)
fig3.update_layout(
    title="Monthly SIP Inflows — Jan 2022 to Dec 2025 (₹ Crore)",
    xaxis_title="Month", yaxis_title="SIP Inflow (₹ Crore)",
    template=PLOTLY_TEMPLATE, height=500,
)
fig3.write_image(str(CHARTS_DIR / "03_sip_trend.png"), width=1400, height=500)
fig3.show()
print("✅ Saved: 03_sip_trend.png")

In [ ]:
# Chart 4 — Category Inflow Heatmap (Seaborn)
cat_inflows["month_label"] = cat_inflows["month"].dt.strftime("%b-%y")
heatmap_data = cat_inflows.pivot_table(
    index="category", columns="month_label", values="net_inflow_crore", aggfunc="sum"
)
month_order = cat_inflows.sort_values("month")["month_label"].unique()
heatmap_data = heatmap_data.reindex(columns=[m for m in month_order if m in heatmap_data.columns])

fig4, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(heatmap_data, annot=True, fmt=".0f", cmap="RdYlGn",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Net Inflow (₹ Crore)"}, annot_kws={"size": 8})
ax.set_title("Category-wise Net Inflow Heatmap (FY 2024-25)", fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Fund Category")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(CHARTS_DIR / "04_category_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: 04_category_heatmap.png")

In [ ]:
# Chart 5 — Investor Demographics (Plotly subplots)
sip_txns    = transactions[transactions["transaction_type"] == "SIP"]
age_counts  = transactions["age_group"].value_counts().reset_index()
age_counts.columns = ["age_group", "count"]
gender_counts = transactions["gender"].value_counts().reset_index()
gender_counts.columns = ["gender", "count"]

fig5 = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Age Group Distribution", "Gender Distribution", "SIP Amount by Age Group"),
    specs=[[{"type": "pie"}, {"type": "pie"}, {"type": "box"}]],
)
fig5.add_trace(go.Pie(labels=age_counts["age_group"], values=age_counts["count"],
    textinfo="label+percent", hole=0.3), row=1, col=1)
fig5.add_trace(go.Pie(labels=gender_counts["gender"], values=gender_counts["count"],
    textinfo="label+percent", hole=0.3,
    marker=dict(colors=["#42A5F5", "#EC407A"])), row=1, col=2)
for age in ["18-25", "26-35", "36-45", "46-55", "56+"]:
    subset = sip_txns[sip_txns["age_group"] == age]["amount_inr"]
    fig5.add_trace(go.Box(y=subset, name=age, boxpoints="outliers"), row=1, col=3)
fig5.update_layout(title_text="Investor Demographics — Age, Gender & SIP Amounts",
    template=PLOTLY_TEMPLATE, height=500, showlegend=False)
fig5.write_image(str(CHARTS_DIR / "05_demographics.png"), width=1400, height=500)
fig5.show()
print("✅ Saved: 05_demographics.png")

In [ ]:
# Chart 6 — Geographic Distribution
state_data = (transactions.groupby("state")["amount_inr"].sum()
    .div(1e7).reset_index().rename(columns={"amount_inr": "total_crore"})
    .sort_values("total_crore", ascending=True))
city_tier = transactions.groupby("city_tier")["amount_inr"].sum().reset_index()

fig6 = make_subplots(rows=1, cols=2,
    subplot_titles=("Total Investment by State (₹ Crore)", "T30 vs B30 City Split"),
    specs=[[{"type": "bar"}, {"type": "pie"}]], column_widths=[0.65, 0.35])
fig6.add_trace(go.Bar(x=state_data["total_crore"], y=state_data["state"],
    orientation="h", marker_color="#42A5F5",
    text=state_data["total_crore"].round(1).astype(str) + " Cr",
    textposition="outside"), row=1, col=1)
fig6.add_trace(go.Pie(labels=city_tier["city_tier"], values=city_tier["amount_inr"],
    textinfo="label+percent", marker=dict(colors=["#FF7043", "#66BB6A"]), hole=0.35), row=1, col=2)
fig6.update_layout(title_text="Geographic Distribution of Investor Transactions",
    template=PLOTLY_TEMPLATE, height=550, showlegend=False)
fig6.write_image(str(CHARTS_DIR / "06_geo_distribution.png"), width=1400, height=550)
fig6.show()
print("✅ Saved: 06_geo_distribution.png")

In [ ]:
# Chart 7 — Folio Count Growth (Plotly Stacked Area)
fig7 = go.Figure()
folio_cols = {"equity_folios_crore": ("Equity", "#1976D2"),
              "debt_folios_crore": ("Debt", "#F57C00"),
              "hybrid_folios_crore": ("Hybrid", "#388E3C"),
              "others_folios_crore": ("Others", "#7B1FA2")}
for col, (label, color) in folio_cols.items():
    fig7.add_trace(go.Scatter(x=folio_count["month"], y=folio_count[col],
        name=label, mode="lines+markers",
        line=dict(color=color, width=2), stackgroup="one"))
fig7.add_annotation(x=folio_count["month"].iloc[0], y=folio_count["total_folios_crore"].iloc[0],
    text=f"📌 {folio_count['total_folios_crore'].iloc[0]} Cr — Jan 2022",
    showarrow=True, arrowhead=2, font=dict(size=11))
fig7.add_annotation(x=folio_count["month"].iloc[-1], y=folio_count["total_folios_crore"].iloc[-1],
    text=f"🏆 {folio_count['total_folios_crore'].iloc[-1]} Cr — Dec 2025",
    showarrow=True, arrowhead=2, font=dict(size=11, color="green"))
fig7.update_layout(title="Mutual Fund Folio Count Growth — Jan 2022 to Dec 2025",
    xaxis_title="Month", yaxis_title="Folios (Crore)",
    template=PLOTLY_TEMPLATE, height=500)
fig7.write_image(str(CHARTS_DIR / "07_folio_growth.png"), width=1400, height=500)
fig7.show()
print("✅ Saved: 07_folio_growth.png")

In [ ]:
# Chart 8 — NAV Return Correlation Matrix (Seaborn)
top10_funds = (nav_history.groupby("amfi_code")["date"].count().nlargest(10).index.tolist())
pivot_returns = (
    nav_history[nav_history["amfi_code"].isin(top10_funds)]
    .pivot_table(index="date", columns="amfi_code", values="daily_return_pct")
    .dropna(thresh=8)   # keep rows with at least 8 non-null values
    .fillna(method="ffill")
    .dropna()
)
code_to_name = (fund_master.set_index("amfi_code")["scheme_name"]
    .str.replace(r" - (Regular|Direct) Plan.*", "", regex=True).str.slice(0, 20))
pivot_returns.columns = [code_to_name.get(c, str(c)) for c in pivot_returns.columns]
corr_matrix = pivot_returns.corr()

fig8, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
    mask=mask, ax=ax, linewidths=0.5, vmin=-1, vmax=1,
    cbar_kws={"label": "Pearson Correlation"})
ax.set_title("NAV Daily Return Correlation Matrix — 10 Funds", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(CHARTS_DIR / "08_correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Saved: 08_correlation_matrix.png")

In [ ]:
# Chart 9 — Sector Allocation Donut
sector_agg = (portfolio.groupby("sector")["market_value_cr"].sum()
    .sort_values(ascending=False).reset_index())
fig9 = go.Figure(go.Pie(
    labels=sector_agg["sector"], values=sector_agg["market_value_cr"],
    hole=0.45, textinfo="label+percent",
    hovertemplate="<b>%{label}</b><br>₹%{value:,.2f} Cr<br>%{percent}<extra></extra>",
    marker=dict(colors=px.colors.qualitative.Set3),
))
fig9.update_layout(title="Equity Fund Portfolio — Sector Allocation by Market Value",
    template=PLOTLY_TEMPLATE, height=550,
    annotations=[dict(text="Sector<br>Weights", x=0.5, y=0.5, showarrow=False, font_size=14)])
fig9.write_image(str(CHARTS_DIR / "09_sector_allocation.png"), width=1000, height=550)
fig9.show()
print("✅ Saved: 09_sector_allocation.png")

# Chart 10 — Fund House Scheme Count
fh_counts = fund_master["fund_house"].value_counts().reset_index()
fh_counts.columns = ["fund_house", "count"]
fig10 = px.bar(fh_counts, x="count", y="fund_house", orientation="h", color="count",
    color_continuous_scale="Blues", title="Number of Schemes per Fund House",
    labels={"count": "Schemes", "fund_house": ""}, template=PLOTLY_TEMPLATE, text="count")
fig10.update_traces(textposition="outside")
fig10.update_layout(height=450, showlegend=False)
fig10.write_image(str(CHARTS_DIR / "10_fund_house_dist.png"), width=1000, height=450)
fig10.show()
print("✅ Saved: 10_fund_house_dist.png")

In [ ]:
# Charts 11–18 — Additional EDA

# Chart 11 — Risk Grade Distribution
risk_counts = fund_master["risk_category"].value_counts().reset_index()
risk_counts.columns = ["risk_category", "count"]
fig11 = px.pie(risk_counts, names="risk_category", values="count",
    title="Fund Risk Grade Distribution", color_discrete_sequence=px.colors.sequential.RdBu,
    hole=0.35, template=PLOTLY_TEMPLATE)
fig11.update_traces(textinfo="label+percent+value")
fig11.write_image(str(CHARTS_DIR / "11_risk_distribution.png"), width=800, height=500)
fig11.show()

# Chart 12 — Expense Ratio Distribution
fig12, ax = plt.subplots(figsize=(10, 5))
sns.histplot(fund_master["expense_ratio_pct"], bins=15, kde=True, color="#5C6BC0", ax=ax, edgecolor="white")
ax.axvline(fund_master["expense_ratio_pct"].mean(), color="red", linestyle="--",
    label=f"Mean: {fund_master['expense_ratio_pct'].mean():.2f}%")
ax.axvline(1.0, color="orange", linestyle=":", label="1% Threshold")
ax.set_title("Expense Ratio Distribution Across All Schemes", fontsize=13, fontweight="bold")
ax.set_xlabel("Expense Ratio (%)"); ax.set_ylabel("Frequency"); ax.legend()
plt.tight_layout()
plt.savefig(CHARTS_DIR / "12_expense_ratio.png", dpi=150, bbox_inches="tight")
plt.show()

# Chart 13 — Transaction Type Distribution
txn_type = transactions["transaction_type"].value_counts().reset_index()
txn_type.columns = ["transaction_type", "count"]
fig13 = px.pie(txn_type, names="transaction_type", values="count",
    title="Investor Transaction Type Distribution",
    color_discrete_sequence=["#42A5F5", "#66BB6A", "#EF5350"],
    hole=0.4, template=PLOTLY_TEMPLATE)
fig13.update_traces(textinfo="label+percent+value")
fig13.write_image(str(CHARTS_DIR / "13_transaction_types.png"), width=800, height=500)
fig13.show()

# Chart 14 — Risk-Return Scatter
fig14 = px.scatter(performance, x="sharpe_ratio", y="return_3yr_pct",
    color="category", size="aum_crore", hover_name="scheme_name",
    hover_data={"fund_house": True, "alpha": True},
    title="Risk-Return Map — Sharpe Ratio vs 3-Year Return",
    labels={"sharpe_ratio": "Sharpe Ratio", "return_3yr_pct": "3-Yr CAGR (%)"},
    template=PLOTLY_TEMPLATE, size_max=30)
fig14.add_hline(y=performance["return_3yr_pct"].mean(), line_dash="dot",
    annotation_text="Avg Return", line_color="gray")
fig14.add_vline(x=1.0, line_dash="dot", annotation_text="Sharpe=1.0", line_color="orange")
fig14.write_image(str(CHARTS_DIR / "14_risk_return_scatter.png"), width=1200, height=600)
fig14.show()

# Chart 15 — Monthly Transaction Trend
txn_monthly = (transactions.groupby(transactions["transaction_date"].dt.to_period("M"))
    .agg(count=("amount_inr", "count"), total_cr=("amount_inr", lambda x: x.sum() / 1e7))
    .reset_index())
txn_monthly["transaction_date"] = txn_monthly["transaction_date"].dt.to_timestamp()
fig15 = make_subplots(specs=[[{"secondary_y": True}]])
fig15.add_trace(go.Bar(x=txn_monthly["transaction_date"], y=txn_monthly["count"],
    name="Count", marker_color="#90CAF9", opacity=0.8), secondary_y=False)
fig15.add_trace(go.Scatter(x=txn_monthly["transaction_date"], y=txn_monthly["total_cr"],
    name="Value (₹Cr)", line=dict(color="#F4511E", width=2.5), mode="lines+markers"), secondary_y=True)
fig15.update_layout(title="Monthly Transaction Volume & Value Trend",
    template=PLOTLY_TEMPLATE, height=500)
fig15.write_image(str(CHARTS_DIR / "15_monthly_transactions.png"), width=1400, height=500)
fig15.show()

# Chart 16 — Category Treemap
cat_dist = fund_master.groupby(["category", "sub_category"]).size().reset_index(name="count")
fig16 = px.treemap(cat_dist, path=["category", "sub_category"], values="count",
    title="Fund Category & Sub-Category Treemap",
    color="count", color_continuous_scale="Viridis", template=PLOTLY_TEMPLATE)
fig16.update_layout(height=550)
fig16.write_image(str(CHARTS_DIR / "16_category_treemap.png"), width=1200, height=550)
fig16.show()

# Chart 17 — Benchmark Comparison
fig17 = px.line(benchmark, x="date", y="close_value", color="index_name",
    title="Benchmark Index Performance Comparison (2022–2026)",
    labels={"close_value": "Index Value", "index_name": "Index"},
    template=PLOTLY_TEMPLATE)
fig17.update_layout(height=500)
fig17.write_image(str(CHARTS_DIR / "17_benchmark_comparison.png"), width=1400, height=500)
fig17.show()

# Chart 18 — Top 10 Funds by AUM
top10_aum = performance.nlargest(10, "aum_crore").copy()
top10_aum["short_name"] = top10_aum["scheme_name"].str.replace(r" - (Regular|Direct).*", "", regex=True).str.slice(0, 30)
fig18 = px.bar(top10_aum.sort_values("aum_crore"), x="aum_crore", y="short_name",
    orientation="h", color="return_3yr_pct", color_continuous_scale="RdYlGn",
    title="Top 10 Funds by AUM — Coloured by 3-Yr Return",
    labels={"aum_crore": "AUM (₹ Crore)", "short_name": ""}, template=PLOTLY_TEMPLATE)
fig18.update_layout(height=550)
fig18.write_image(str(CHARTS_DIR / "18_top10_aum.png"), width=1200, height=550)
fig18.show()

print("\n✅ All 18 charts saved to reports/charts/")

In [ ]:
# Statistical Analysis
print("=" * 70)
print("  STATISTICAL ANALYSIS — NAV DAILY RETURNS")
print("=" * 70)
returns = nav_history["daily_return_pct"].dropna()
print(f"  Count     : {len(returns):,}")
print(f"  Mean      : {returns.mean():.4f}%")
print(f"  Median    : {returns.median():.4f}%")
print(f"  Std Dev   : {returns.std():.4f}%")
print(f"  Skewness  : {returns.skew():.4f}  ({'right-skewed' if returns.skew() > 0 else 'left-skewed'})")
print(f"  Kurtosis  : {returns.kurtosis():.4f}")
print(f"  Min / Max : {returns.min():.4f}% / {returns.max():.4f}%")
print(f"  Missing   : {nav_history['daily_return_pct'].isna().mean()*100:.1f}%")

print("\n  SCHEME PERFORMANCE METRICS")
metrics = ["return_1yr_pct", "return_3yr_pct", "sharpe_ratio", "alpha", "beta"]
print(performance[metrics].describe().round(3).to_string())

print("\n  OUTLIER DETECTION — SIP Amounts (IQR)")
sip_amounts = transactions[transactions["transaction_type"] == "SIP"]["amount_inr"]
Q1, Q3 = sip_amounts.quantile(0.25), sip_amounts.quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
outliers = sip_amounts[(sip_amounts < lower) | (sip_amounts > upper)]
print(f"  IQR: ₹{IQR:,.0f} | Lower: ₹{lower:,.0f} | Upper: ₹{upper:,.0f}")
print(f"  Outliers: {len(outliers):,} ({len(outliers)/len(sip_amounts)*100:.1f}%)")

In [ ]:
# 10 Key Findings Summary
findings = [
    ("F1", "SIP inflows grew 169% in 4 years", "₹11,517 Cr (Jan 2022) → ₹31,002 Cr (Dec 2025)"),
    ("F2", "SBI dominates AUM at ₹12.5 lakh crore", "2x the AUM of 3rd-ranked HDFC MF"),
    ("F3", "Folios doubled: 13.26 Cr → 26.12 Cr", "Millions of new investors entering markets"),
    ("F4", "26-35 age group drives SIP adoption", "Digital-first segment, largest transaction cohort"),
    ("F5", "B30 cities contribute ~35% of investment", "Financial inclusion initiatives working"),
    ("F6", "Banking & Utilities dominate portfolios", "High BFSI concentration = interest rate risk"),
    ("F7", "High equity fund correlation >0.80", "Cross-AMC diversification in same category adds little value"),
    ("F8", "Mid Cap & Small Cap see highest inflows", "Increased risk appetite post-2022 recovery"),
    ("F9", "Funds with Sharpe >1.0 in equity category", "Direct plans consistently beat regular plans"),
    ("F10", "NAV daily returns are left-skewed", "More frequent small gains — reinforces SIP logic"),
]
print(f"{'#':<4} {'Finding':<45} {'Interpretation'}")
print("-" * 100)
for fid, finding, interpretation in findings:
    print(f"{fid:<4} {finding:<45} {interpretation}")